![image_1781085979001.png](./image_1781085979001.png "image_1781085979001.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
# Initialize Spark session
spark = SparkSession.builder.appName("OrderItemsDF").getOrCreate()

# Dataset
data = [
    (20, 301, 5),
    (21, 301, 7),
    (22, 302, 5),
    (23, 302, 7),
    (24, 303, 5),
    (25, 303, 9),
    (26, 304, 7),
    (27, 304, 9),
    (28, 305, 5),
    (29, 305, 7),
    (30, 306, 5),
    (31, 306, 9)
]

# Define schema (column names)
columns = ["id", "order_id", "product_id"]

# Create DataFrame
order_items_df = spark.createDataFrame(data, columns)

# Show DataFrame
order_items_df.show()


In [0]:
os_df = order_items_df.alias("os")
ot_df = order_items_df.alias("ot")

result_df = (
    os_df.join(ot_df, os_df.order_id == ot_df.order_id)
    .filter(os_df.product_id < ot_df.product_id)
    .groupBy(os_df.product_id, ot_df.product_id)
    .agg(f.count("*").alias("times_bought_together"))
    .select(
        os_df.product_id.alias("product_id_1"),
        ot_df.product_id.alias("product_id_2"),
        f.col("times_bought_together"),
    )
    .filter(f.col("times_bought_together") > 1)
    .orderBy(f.desc(f.col("times_bought_together")), f.col("product_id_1"))
)
display(result_df)